In [1]:
import numpy as np  
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset , DataLoader

from nn_live import Visualizer

import os

from sklearn.model_selection import train_test_split


In [2]:
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "pip", "install", "-e",
#     r"C:\Users\ankit\Desktop\pytorch\DL_Visulization_tool"])


In [3]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name())
else:
    print("CPU")

NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zalando-research/fashionmnist")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\ankit\.cache\kagglehub\datasets\zalando-research\fashionmnist\versions\4


In [5]:
os.listdir(path)

['fashion-mnist_test.csv',
 'fashion-mnist_train.csv',
 't10k-images-idx3-ubyte',
 't10k-labels-idx1-ubyte',
 'train-images-idx3-ubyte',
 'train-labels-idx1-ubyte']

In [6]:
path_train=os.path.join(path, "fashion-mnist_train.csv")
path_test=os.path.join(path, "fashion-mnist_test.csv")

In [7]:
# set random Seeds dor reproducibility
torch.manual_seed(42)

In [8]:
df=pd.read_csv(path_train)
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
X = df.drop('label',axis=1)
y= df['label']

In [10]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

- Why divide by 255?<br>
    - Images store pixel intensity from 0–255 (uint8)<br>
    - Neural networks work better with small, normalized values<br>
    - Prevents unstable gradients and slow learning

In [11]:

 # scaling the feautures
X_train = X_train/255.0
X_test = X_test/255.0

In [12]:
# create CustomDataset Class
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = torch.tensor(features.to_numpy(), dtype=torch.float32)
    self.labels = torch.tensor(labels.to_numpy(), dtype=torch.long)

  def __len__(self):

    return len(self.features)

  def __getitem__(self, index):

    return self.features[index], self.labels[index]



In [13]:
train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)


In [14]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)


In [15]:
class MyNN(nn.Module):

    def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer):

        super().__init__()

        layers = []

        for i in range(num_hidden_layers):
            layers.append(nn.Linear(input_dim, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=0.3))
            input_dim = neurons_per_layer
        
        layers.append(nn.Linear(neurons_per_layer, output_dim))

        self.model = nn.Sequential(*layers)  # *layers unpack the list

    def forward(self, x):

        return self.model(x)

In [20]:
# objective function

def objective(trial):

    # next hyperparameter values from the search space
    num_hidden_layers = trial.suggest_int("_hidden_layers", 1, 5)
    neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 120, step=8)

    # moden init
    input_dim =784
    output_dim =10

    model =MyNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer)
    model.to(device='cuda')
    
    # params
    learning_rate = 0.1
    epoch = 10

    # loss function
    criterion = nn.CrossEntropyLoss()

    # optimizer
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)


    # training loop
    viz = Visualizer(model, port=5000)

    for epoch in range(epoch):

        for batch_features, batch_labels in train_loader:

            # move data to gpu
            batch_features = batch_features.to(device='cuda')
            batch_labels = batch_labels.to(device='cuda')

            # forward pass
            outputs = model(batch_features)

            # calculate loss
            loss = criterion(outputs, batch_labels)

            # back pass
            optimizer.zero_grad()
            loss.backward()

            # update grads
            optimizer.step()

            viz.step(epoch=epoch+1, loss=loss)

    # evaluation
    model.eval()

    total = 0
    correct = 0

    with torch.no_grad():

        for batch_features, batch_labels in test_loader:

                # move data to gpu
            batch_features = batch_features.to(device='cuda')
            batch_labels = batch_labels.to(device='cuda')

            outputs = model(batch_features)

            _, predicted = torch.max(outputs, 1)

            total = total + batch_labels.shape[0]

            correct = correct + (predicted == batch_labels).sum().item()

        accuracy=correct/total

    return accuracy
            


In [21]:
# !pip install optuna

In [22]:
import optuna

study = optuna.create_study(direction='maximize')

[I 2026-05-03 22:51:55,992] A new study created in memory with name: no-name-db4586ce-01c5-4095-b0f7-252335f17136


In [ ]:
study.optimize(objective, n_trials=3)